# RuleShift Benchmark v1 — Kaggle Benchmark

Official `kaggle_benchmarks` notebook for the RuleShift Benchmark v1 cognitive flexibility benchmark.

- **Leaderboard task:** `ruleshift_benchmark_v1_binary` — Binary (post-shift probe accuracy); evaluated over `public_leaderboard` and `private_leaderboard`
- **Supplementary:** Narrative — same leaderboard episodes, not leaderboard-scored; results included in the payload as robustness evidence only
- **Local only:** `dev` split — used for local validation and debug checks; never included in the official leaderboard evaluation
- **Metric:** Post-shift Probe Accuracy = correct probes / total probes
- **Per-episode return:** `(num_correct, 4)`

## Imports

In [ ]:
import sys
from pathlib import Path

import kaggle_benchmarks as kbench

## Setup

Resolves the repo root and adds `src/` to `sys.path`. On Kaggle, the benchmark repo is attached as an input dataset. The private evaluation dataset is a separate authorized attachment that is discovered later only when present.

In [ ]:
def _find_repo_root() -> Path:
    """Locate the repo root from the notebook's working directory or Kaggle input paths."""
    candidates: list[Path] = []
    seen: set[Path] = set()

    for origin in (Path.cwd().resolve(),):
        for candidate in (origin, *origin.parents):
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for search_root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not search_root.exists():
            continue
        for manifest_path in search_root.rglob("frozen_artifacts_manifest.json"):
            candidate = manifest_path.parents[2]
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for candidate in candidates:
        if (candidate / "src").is_dir() and (
            candidate / "packaging" / "kaggle" / "frozen_artifacts_manifest.json"
        ).is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate repo root. Expected src/ and "
        "packaging/kaggle/frozen_artifacts_manifest.json."
    )


REPO_ROOT = _find_repo_root()

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

PRIVATE_DATASET_ROOT = None

## Load frozen splits

`dev` and `public_leaderboard` are always reconstructed from frozen seed manifests. `private_leaderboard` is discovered lazily and loaded only when the attached authorized private dataset mount is present.

In [ ]:
from core.kaggle import BinaryResponse, Label, normalize_binary_response, normalize_narrative_response, score_episode
from core.kaggle import NotebookStatus
from core.splits import load_frozen_split
from core.private_split import discover_private_dataset_root, load_private_split
from core.parser import PARSER_VERSION
from core.metrics import METRIC_VERSION
from tasks.ruleshift_benchmark.render import render_binary_prompt, render_narrative_prompt

_DEV_PARTITION = "dev"
_LEADERBOARD_PARTITIONS = ("public_leaderboard", "private_leaderboard")
_PUBLIC_RUNTIME_PARTITIONS = (_DEV_PARTITION, "public_leaderboard")
_PROGRESS_REFRESH_EVERY = 10
_bootstrap_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)

_bootstrap_status.start(
    "Notebook bootstrap",
    detail="Locating repo root and frozen split artifacts",
)

frozen_splits: dict = {
    partition: load_frozen_split(partition)
    for partition in _PUBLIC_RUNTIME_PARTITIONS
}

_setup_warnings = 0
try:
    PRIVATE_DATASET_ROOT = discover_private_dataset_root()
except FileNotFoundError as _private_err:
    PRIVATE_DATASET_ROOT = None
    _setup_warnings += 1
    _private_dataset_detail = f"private dataset unavailable: {_private_err}"
else:
    if PRIVATE_DATASET_ROOT is not None:
        frozen_splits["private_leaderboard"] = load_private_split(PRIVATE_DATASET_ROOT)
        _private_dataset_detail = f"private dataset attached: {PRIVATE_DATASET_ROOT}"
    else:
        _private_dataset_detail = "private dataset not attached; private_leaderboard skipped"

_partition_summary = ", ".join(
    f"{partition}={len(records)}" for partition, records in frozen_splits.items()
)
_bootstrap_status.finish(
    detail=(
        f"repo root: {REPO_ROOT}\n"
        f"parser={PARSER_VERSION} | metric={METRIC_VERSION}\n"
        f"{_private_dataset_detail}\n"
        f"splits: {_partition_summary}"
    ),
    processed=len(frozen_splits),
    total=len(_PUBLIC_RUNTIME_PARTITIONS) + 1,
    warnings=_setup_warnings,
)

## Build evaluation dataframes

Builds two separate dataframes:

- `dev_df` — dev split only; local validation, never submitted.
- `leaderboard_df` — `public_leaderboard` and `private_leaderboard` only; fed to the official evaluation path.

In [ ]:
import pandas as pd


_dataframe_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_dataframe_status.start(
    "Building evaluation frames",
    detail="Preparing dev and leaderboard dataframes",
)


def _build_rows(partition: str, records) -> list[dict]:
    rows = []
    for record in records:
        ep = record.episode
        rows.append({
            "episode_id": ep.episode_id,
            "split": partition,
            "difficulty": ep.difficulty.value,
            "template_id": ep.template_id.value,
            "template_family": ep.template_family.value,
            "shift_position": str(ep.shift_after_position),
            "transition_type": ep.transition.value,
            "prompt_binary": render_binary_prompt(ep),
            "prompt_narrative": render_narrative_prompt(ep),
            "probe_targets": tuple(t.value for t in ep.probe_targets),
        })
    return rows


_dev_rows: list[dict] = _build_rows(_DEV_PARTITION, frozen_splits[_DEV_PARTITION])
dev_df = pd.DataFrame(_dev_rows)
assert set(dev_df["split"]) == {_DEV_PARTITION}

_available_leaderboard_partitions = tuple(
    p for p in _LEADERBOARD_PARTITIONS if p in frozen_splits
)
leaderboard_rows = [
    row
    for _partition in _available_leaderboard_partitions
    for row in _build_rows(_partition, frozen_splits[_partition])
]
leaderboard_df = pd.DataFrame(leaderboard_rows)
assert set(leaderboard_df["split"]) == set(_available_leaderboard_partitions)

_dataframe_status.finish(
    detail=(
        f"dev_df rows: {len(dev_df)}\n"
        f"leaderboard_df rows: {len(leaderboard_df)}\n"
        f"leaderboard partitions: {', '.join(_available_leaderboard_partitions)}"
    ),
    processed=len(dev_df) + len(leaderboard_df),
    total=len(dev_df) + len(leaderboard_df),
)
leaderboard_df[["episode_id", "split", "difficulty", "template_id", "template_family", "shift_position", "transition_type"]].head()

## Binary task — official leaderboard task

Registered with `@kbench.task`. `%choose ruleshift_benchmark_v1_binary` selects this as the leaderboard entry. At evaluation time, kbench calls this function once per row in `evaluation_data`, injecting `llm` and matching remaining columns by name. Returns `(num_correct, 4)` per episode.

In [ ]:
@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
)
def ruleshift_benchmark_v1_binary(
    llm,
    prompt_binary: str,
    probe_targets: tuple,
    **kwargs,
) -> tuple[int, int]:
    try:
        response = llm.prompt(
            prompt_binary,
            schema=BinaryResponse,
        )
        predictions = normalize_binary_response(response)
    except Exception:
        predictions = None

    return score_episode(predictions, probe_targets)

## Narrative function — supplementary companion

Not a `@kbench.task`; not selectable via `%choose`. Results are collected manually and included in the payload as robustness evidence only — they do not affect the leaderboard score.

In [ ]:
def ruleshift_benchmark_v1_narrative(
    llm,
    prompt_narrative: str,
    probe_targets: tuple,
) -> tuple[int, int]:
    try:
        response = llm.prompt(prompt_narrative)
        predictions = normalize_narrative_response(response)
    except Exception:
        predictions = None
    return score_episode(predictions, probe_targets)

## Local: scoring sanity checks

Verifies parser and scorer against known inputs using one row from `dev_df`. Local only — does not run on the Kaggle evaluation path.

In [ ]:
sample_row = dev_df.iloc[0]
print(f"Episode: {sample_row['episode_id']}")
print(f"Split: {sample_row['split']}")
print(f"Targets: {sample_row['probe_targets']}")
print()

# Verify scoring with a perfect prediction
perfect = score_episode(sample_row["probe_targets"], sample_row["probe_targets"])
print(f"Perfect prediction score: {perfect}")
assert perfect == (4, 4), f"Expected (4, 4), got {perfect}"

# Verify scoring with an invalid prediction
invalid = score_episode(None, sample_row["probe_targets"])
print(f"Invalid prediction score: {invalid}")
assert invalid == (0, 4), f"Expected (0, 4), got {invalid}"

# Verify normalize with a raw text response
targets_text = ", ".join(sample_row["probe_targets"])
normalized = normalize_binary_response(targets_text)
print(f"Parsed '{targets_text}' -> {normalized}")
assert normalized == sample_row["probe_targets"], f"Parse mismatch"

# Verify structured response path
labels = [Label(t) for t in sample_row["probe_targets"]]
structured = BinaryResponse(*labels)
structured_norm = normalize_binary_response(structured)
print(f"Structured response -> {structured_norm}")
assert structured_norm == sample_row["probe_targets"], f"Structured mismatch"

# Verify malformed text
malformed = normalize_binary_response("I don't know")
malformed_score = score_episode(malformed, sample_row["probe_targets"])
print(f"Malformed score: {malformed_score}")
assert malformed_score == (0, 4), f"Expected (0, 4), got {malformed_score}"

print("\nAll local checks passed.")

## Local: dev split validation

Runs Binary and Narrative against `dev_df` only. Results are stored in `_dev_*` variables and do not touch `binary_results`, `narrative_results_df`, or the canonical payload. Safe to skip.

In [ ]:
_dev_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_dev_total = len(dev_df)
_dev_status.start(
    "Dev validation",
    detail="Running Binary and Narrative on dev only",
    processed=0,
    total=_dev_total * 2,
)

try:
    _dev_binary_results = ruleshift_benchmark_v1_binary.evaluate(
        llm=[kbench.llm],
        evaluation_data=dev_df,
    )
    _dev_binary_df = _dev_binary_results.as_dataframe()
    _dev_status.update(
        detail="Binary validation complete; Narrative validation in progress",
        processed=len(_dev_binary_df),
        total=_dev_total * 2,
        force=True,
    )
    display(_dev_binary_df.head())
except Exception as _dev_binary_exc:
    _dev_status.update(
        detail=f"Binary validation skipped: {_dev_binary_exc}",
        processed=0,
        total=_dev_total * 2,
        warnings=1,
        force=True,
    )
    _dev_binary_results = None

try:
    _dev_nar_rows: list[dict] = []
    for _idx, (_, _row) in enumerate(dev_df.iterrows(), start=1):
        _nc, _tot = ruleshift_benchmark_v1_narrative(
            kbench.llm,
            _row["prompt_narrative"],
            _row["probe_targets"],
        )
        _dev_nar_rows.append({"num_correct": _nc, "total": _tot})
        _dev_status.update(
            detail="Narrative validation in progress",
            processed=_dev_total + _idx,
            total=_dev_total * 2,
        )
    _dev_narrative_df = pd.DataFrame(_dev_nar_rows)
    _dev_status.finish(
        detail=(
            f"Binary rows: {0 if _dev_binary_results is None else len(_dev_binary_df)}\n"
            f"Narrative rows: {len(_dev_narrative_df)}"
        ),
        processed=_dev_total * 2,
        total=_dev_total * 2,
    )
    display(_dev_narrative_df.head())
except Exception as _dev_narrative_exc:
    _dev_status.finish(
        detail=f"Narrative validation skipped: {_dev_narrative_exc}",
        processed=_dev_total,
        total=_dev_total * 2,
        warnings=1,
    )
    _dev_narrative_df = None

## Official Binary evaluation

Calls `.evaluate()` with `leaderboard_df` as `evaluation_data` (all leaderboard partitions). `split` is dropped before passing in so task logic cannot depend on it.

In [ ]:
_binary_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_binary_status.start(
    "Official Binary evaluation",
    detail="Running official Binary task over leaderboard episodes",
    processed=0,
    total=len(leaderboard_df),
)

_binary_eval_df = leaderboard_df.drop(columns=["split"])
binary_results = ruleshift_benchmark_v1_binary.evaluate(
    llm=[kbench.llm],
    evaluation_data=_binary_eval_df,
)
_binary_results_df = binary_results.as_dataframe()
_binary_status.finish(
    detail=f"Binary rows produced: {len(_binary_results_df)}",
    processed=len(_binary_results_df),
    total=len(leaderboard_df),
)

## Official Narrative evaluation

Runs the Narrative companion over `leaderboard_df`. Non-blocking: a failure sets `narrative_results_df = None` without preventing the Binary result from being emitted.

In [ ]:
_narrative_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_narrative_status.start(
    "Official Narrative evaluation",
    detail="Running supplementary Narrative evaluation",
    processed=0,
    total=len(leaderboard_df),
)

try:
    _nar_eval_df = leaderboard_df[["prompt_narrative", "probe_targets"]]
    _nar_rows: list[dict] = []
    _nar_total = len(_nar_eval_df)
    for _idx, (_, _row) in enumerate(_nar_eval_df.iterrows(), start=1):
        _nc, _tot = ruleshift_benchmark_v1_narrative(
            kbench.llm,
            _row["prompt_narrative"],
            _row["probe_targets"],
        )
        _nar_rows.append({"num_correct": _nc, "total": _tot})
        _narrative_status.update(
            detail="Collecting Narrative episode scores",
            processed=_idx,
            total=_nar_total,
        )
    narrative_results_df = pd.DataFrame(_nar_rows)
    _narrative_status.finish(
        detail=f"Narrative rows produced: {len(narrative_results_df)}",
        processed=len(narrative_results_df),
        total=_nar_total,
    )
except Exception as _narrative_exc:
    _narrative_status.finish(
        detail=f"Narrative evaluation skipped: {_narrative_exc}",
        processed=0,
        total=len(leaderboard_df),
        warnings=1,
    )
    narrative_results_df = None

## Canonical payload

Assembles and validates the official result payload from `binary_results` and `narrative_results_df`. The `print()` output is the authoritative Kaggle-emitted artifact. Both inputs are required — missing either raises an error.

In [ ]:
import json
from core.kaggle import (
    build_kaggle_payload,
    normalize_count_result_df,
    validate_kaggle_payload,
)

_payload_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_payload_status.start(
    "Canonical payload",
    detail="Normalizing evaluation outputs and validating final artifact",
    processed=0,
    total=3,
)

if 'binary_results' not in locals() or binary_results is None:
    raise ValueError(
        "CRITICAL: binary_results is missing. "
        "Binary evaluation did not run or failed before producing results."
    )

if 'narrative_results_df' not in locals() or narrative_results_df is None:
    raise ValueError(
        "CRITICAL: narrative_results_df is missing. "
        "Narrative evaluation is mandatory for a valid release. "
        "Both Binary and Narrative must run on the same frozen episodes."
    )

binary_df_payload = normalize_count_result_df(
    binary_results.as_dataframe(),
    dataframe_label="binary_df",
)
narrative_df_payload = normalize_count_result_df(
    narrative_results_df,
    dataframe_label="narrative_df",
)
_payload_status.update(
    detail="Normalized Binary and Narrative outputs",
    processed=1,
    total=3,
    force=True,
)

payload = build_kaggle_payload(
    binary_df=binary_df_payload,
    narrative_df=narrative_df_payload,
)
_payload_status.update(
    detail="Payload assembled; running contract validation",
    processed=2,
    total=3,
    force=True,
)
validate_kaggle_payload(payload)
_payload_status.finish(
    detail=(
        f"Validated payload with {payload['primary_result']['total_episodes']} episodes\n"
        f"binary score: {payload['primary_result']['score']:.4f} | narrative score: {payload['narrative_result']['score']:.4f}"
    ),
    processed=3,
    total=3,
)

# Public result: the print output below is the authoritative Kaggle-emitted artifact.
print("=== RuleShift Benchmark v1 — Canonical Result ===")
print(json.dumps(payload, indent=2))
print("=== End Canonical Result ===")

# Optional debug side file — not the authoritative result.
try:
    with open("benchmark_result.json", "w") as _f:
        json.dump(payload, _f, indent=2)
except Exception:
    pass

## Task selection

Selects `ruleshift_benchmark_v1_binary` as the single leaderboard task. Kaggle Benchmarks uses `%choose` to determine which task result is submitted.

In [ ]:
%choose ruleshift_benchmark_v1_binary